# Structured Output and Model-Agnostic Architecture

Getting reliable JSON out of a stochastic model, and building so any model can be swapped in.

This notebook is an original tutorial extending the course with current
(2026) state-of-the-art practice, interview-focused.

## Learning Objectives

- Explain how JSON-schema enforcement actually works (constrained decoding vs instruction-following).
- Contrast OpenAI strict structured outputs with Anthropic's approach.
- Implement the validate-and-retry pattern and explain why it is still needed.
- Design a model-agnostic abstraction layer: provider interface, capability flags, task-tier routing.
- Explain prompt-portability pitfalls and eval-gated model swaps.

## Mental Model

Two separate guarantees are commonly conflated:

- **Syntactic validity** ("it parses against the schema") — this can be
  *guaranteed* by constrained decoding: the schema compiles to a grammar and
  invalid tokens are masked at every decoding step.
- **Semantic correctness** ("the values are right") — no decoder can guarantee
  this. A perfectly-shaped JSON object can still contain wrong answers.

Constrained decoding buys you the first; validation, evals, and judges buy
you the second. Similarly, "model-agnostic" does not mean one prompt runs
everywhere — it means the *interfaces and routing* are model-independent while
prompts stay versioned per task with per-provider overrides.

## Important Concepts

- **Constrained decoding**: schema → grammar/finite-state machine → token masking each step (OpenAI `strict: true`, Outlines/XGrammar/llguidance for open models, GBNF in llama.cpp).
- **Instruction-based structure**: Anthropic historically relied on tool schemas + strong instruction following; `output_format` / strict tool use added later — a standard interview contrast.
- **Validate-and-retry ("re-ask")**: parse with Pydantic/Zod; on failure, re-prompt with the validation error; bounded retries with fallback (Instructor, Vercel AI SDK `generateObject`).
- **Provider interface + capability flags**: adapters declare max context, schema support, logprobs, vision/audio, caching; route by required capability.
- **Task-tier routing**: config maps tasks to tiers (`extraction: cheap-fast`, `synthesis: frontier`), tiers to current models — a swap is a config change.
- **Gateways**: LiteLLM (self-host proxy, OpenAI format for 100+ providers), OpenRouter (managed, +25-40ms), Vercel AI SDK (TypeScript app layer).

## Need-To-Know Coverage Checklist

- [x] How strict mode guarantees parse-ability (grammar-constrained token masking).
- [x] Schema subset restrictions in strict mode (all fields required, `additionalProperties: false`).
- [x] Why validation + retry is needed even with strict mode.
- [x] Failure modes: right shape wrong values, quality loss under tight constraints, enum abuse, refusal fields.
- [x] Reason-then-extract two-pass pattern to protect reasoning quality.
- [x] LiteLLM vs OpenRouter vs Vercel AI SDK; where each sits in the stack.
- [x] Prompt portability pitfalls across OpenAI/Anthropic/Gemini.
- [x] Eval-gated swaps: no model change ships without a golden-dataset delta.

## Deep Study Notes

### How schema enforcement works

- **OpenAI Structured Outputs (`strict: true`)**: the JSON schema is compiled
  into a grammar; at each decoding step, tokens that would violate the grammar
  are masked out. Syntactic validity is guaranteed. Restrictions apply: all
  fields required, `additionalProperties: false`, limited schema keywords.
  Available on both `response_format` and tool definitions.
- **Anthropic**: structured outputs via `output_format` and strict tool use
  (since late 2025); historically tool-schema + instruction-following — very
  reliable but not token-level guaranteed. Interviewers like this contrast.
- **Open models**: Outlines, XGrammar, llguidance, llama.cpp GBNF; vLLM and
  SGLang ship integrated grammar backends.

### Why validate-and-retry survives strict mode

Strict mode guarantees *syntax, not semantics*. You still need:

- **Pydantic/Zod parsing** as the last line (types, ranges, business rules
  like "percentages sum to 100").
- **Re-ask on failure**: re-prompt including the validation error message;
  bounded retries; fallback behavior after N failures.

### Failure modes (the interview list)

- Right shape, wrong values — schema cannot check truth.
- **Quality degradation under constraints**: the "Let Me Speak Freely?"
  research showed tight format restrictions can hurt reasoning. Mitigation:
  a free-text `reasoning` field *before* the answer fields, or a two-pass
  reason-then-extract flow.
- Grammar-forced "safe defaults": when the model wants to say something
  outside the schema, the grammar forces *something* valid — silently wrong.
- Enum abuse: forced to pick a category even when none fits; always include
  an `other`/`unknown` escape hatch.
- Refusals arriving in a separate `refusal` field instead of content.
- Unsupported schema features silently downgrading strict mode to best-effort.

### Model-agnostic architecture

Two layers, both useful:

- **Gateway/router**: **LiteLLM** (open-source proxy, self-hosted, OpenAI
  format for 100+ providers, budgets/keys/fallbacks, ~8ms overhead) or
  **OpenRouter** (managed, widest catalog, zero setup, +25-40ms, no
  governance). Enterprise: Portkey/TrueFoundry.
- **Application SDK**: **Vercel AI SDK** (TypeScript; provider-swappable
  `generateText`/`streamText`/`generateObject`) or a thin in-house provider
  interface. LangChain is now mostly used for ecosystem, not as the
  abstraction layer.

Design patterns:

- **Provider interface + capability flags**: expose `complete()`, `embed()`,
  `classify_structured()`; each adapter declares capabilities (context size,
  strict-schema support, logprobs, audio, caching). Route by required
  capability, never by hard-coded model name.
- **Task-tier routing**: `extraction: cheap-fast`, `taxonomy: frontier`,
  `synthesis: frontier-long-context` in config; tiers map to current models.
- **Prompt portability pitfalls**: system-prompt adherence differs; JSON-mode
  quirks differ; Anthropic favors XML tags, others markdown; tool-call formats
  differ; refusal/hedging behavior differs; temperature semantics differ;
  caching APIs differ. Keep prompts versioned per task with per-provider
  overrides — do not pretend one prompt is universal.
- **Eval-gated swaps**: golden dataset + scorers wired into CI (Promptfoo as
  gate, Braintrust/LangSmith for dashboards); every production failure becomes
  a regression row; no swap ships without a quality delta. Rollout:
  shadow traffic → 1-5% canary → ramp → full.

## Common Failure Modes

- Trusting strict mode to mean "correct" — it only means "parses".
- No `other` option in enums → confidently wrong classifications.
- One shared prompt across providers → silent quality regression on the second provider.
- Hard-coded model names sprinkled through the codebase → a swap is a refactor instead of a config change.
- Swapping models on vibes ("looks better in a few chats") with no eval delta.
- Cramming reasoning into a rigid schema → measurably worse analysis.

## Implementation Notes

- Always define schemas in one place (Pydantic/Zod) and derive JSON schema from them.
- Add a `reasoning` free-text field before answer fields in analysis schemas.
- Log schema-validation failure rates per model — it is an early drift signal.
- Keep a `model_config.yaml`: task → tier → model, reviewed in PRs like code.

In [ ]:
"""Validate-and-retry + task-tier routing, in miniature.

Shows the two patterns without external dependencies. In production the
validator is Pydantic/Zod and the router is LiteLLM/AI SDK config.
"""
import json

MODEL_CONFIG = {
    # task -> tier; tier -> current model. A swap = edit this dict only.
    "tiers": {"cheap-fast": "haiku-4.5", "frontier": "fable-5"},
    "tasks": {"classify_response": "cheap-fast", "synthesize_themes": "frontier"},
}


def route(task: str) -> str:
    return MODEL_CONFIG["tiers"][MODEL_CONFIG["tasks"][task]]


def validate(raw: str) -> tuple[dict | None, str | None]:
    """Schema + business-rule validation. Strict mode guarantees parse,
    but NOT these semantic rules — that's why this layer survives."""
    try:
        data = json.loads(raw)
    except json.JSONDecodeError as e:
        return None, f"invalid JSON: {e}"
    allowed = {"pricing", "onboarding", "support", "other"}
    if data.get("theme") not in allowed:
        return None, f"theme must be one of {sorted(allowed)}"
    if not 0.0 <= data.get("confidence", -1) <= 1.0:
        return None, "confidence must be in [0, 1]"
    return data, None


def call_with_retry(prompt: str, model: str, max_retries: int = 2) -> dict:
    """Re-ask pattern: feed the validation error back to the model."""
    attempts = [  # simulated model outputs: bad enum, then valid
        '{"theme": "cost", "confidence": 0.9}',
        '{"theme": "pricing", "confidence": 0.9}',
    ]
    # Simulated: a real client would send the updated prompt each retry.
    for attempt, raw in enumerate(attempts[: max_retries + 1]):
        data, err = validate(raw)
        if data is not None:
            return {"result": data, "attempts": attempt + 1, "model": model}
        prompt += f"\nYour previous output failed validation: {err}. Fix it."
        print(f"attempt {attempt + 1} failed: {err} -> re-asking")
    raise ValueError("validation failed after retries")


model = route("classify_response")
print(f"routed classify_response -> {model}")
print(json.dumps(call_with_retry("Classify this response.", model), indent=2))


## Practice

1. Add an `evidence_quote` field to the schema and a validation rule that the
   quote must appear verbatim in a given transcript string (anti-hallucination check).
2. Add a capability flag system: mark `haiku-4.5` as lacking `strict_schema`
   and make `route()` fall back to the frontier tier when a task requires it.
3. Explain why a schema with `"category": enum[5 options]` and no `other`
   option is a bug, and what data you would look at to detect it in production.
4. Sketch the four-stage rollout (shadow → canary → ramp → full) for swapping
   the classification model, naming the metrics that gate each stage.

## Design Checklist

- [ ] Schemas defined once (Pydantic/Zod); JSON schema derived, never duplicated.
- [ ] Strict/constrained decoding on where supported; validation layer regardless.
- [ ] Re-ask with the validation error; bounded retries; explicit fallback.
- [ ] Enums include an escape hatch; refusal fields handled.
- [ ] Reasoning field precedes answer fields in analysis schemas.
- [ ] All calls go through a gateway; task-tier routing in config.
- [ ] Prompts versioned per task, with per-provider overrides.
- [ ] Model swaps are eval-gated and rolled out gradually.